## Integración de datos: Cámaras, Bitcarrier y Accidentalidad

Este cuaderno carga y une:
- **Cámaras** (`Camaras_Salvavidas_Bogota.geojson`)
- **Vectores Bitcarrier** (`Vectores Bitcarrier.geojson`) + **Velocidades** (CSV 2019-2020)
- **Accidentalidad** (excels 2018-2022)

Salida: datos limpios y unidos por cámara, segmentos vinculados, velocidades filtradas y accidentes asignados a áreas de cobertura por cámara.


In [1]:
import os
import re
import json
import warnings
from datetime import datetime

import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point, LineString, MultiLineString, box
from shapely.ops import unary_union
from tqdm import tqdm

warnings.filterwarnings("ignore")
pd.options.display.max_columns = 200
pd.options.display.width = 200

print("Versions -> pandas:", pd.__version__, ", geopandas:", gpd.__version__)


Versions -> pandas: 2.3.3 , geopandas: 1.1.1


In [2]:
# Paths (relative to workspace root)
CAMERAS_PATH = "Camaras_Salvavidas_Bogota.geojson"
VECTORES_PATH = "Vectores Bitcarrier.geojson"
VEL_DIR_2019 = os.path.join("Velocidades 2019-2020", "2019")
VEL_DIR_2020 = os.path.join("Velocidades 2019-2020", "2020")
ACC_DIR = "ACCIDENTALIDAD"
OUTPUT_DIR = "outputs"

# Parameters
EPSG_LATLON = 4326
EPSG_DISTANCE = 3116  # Projected for Colombia distances
CAMERA_SEGMENT_BUFFER_M = 60  # associate segments to cameras within this distance
COVERAGE_BUFFER_M = 30        # coverage area around linked segments
MAX_SEGMENTS_PER_CAMERA = 50  # safety cap

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("OUTPUT_DIR:", os.path.abspath(OUTPUT_DIR))


OUTPUT_DIR: c:\Users\davib\Downloads\Camaras Salvavidas\outputs


In [3]:
# Helper utilities (accent-insensitive column matching)
from typing import List, Optional, Tuple
import unicodedata


def normalize_text(value: str) -> str:
    if value is None:
        return ""
    # lower, strip, remove accents/diacritics
    text = unicodedata.normalize("NFKD", str(value).strip().lower())
    return "".join(ch for ch in text if not unicodedata.combining(ch))


def pick_first(cols: List[str], candidates: List[str]) -> Optional[str]:
    if not cols:
        return None
    normalized_cols = [normalize_text(c) for c in cols]
    normalized_candidates = [normalize_text(c) for c in candidates]
    for cand in normalized_candidates:
        if cand in normalized_cols:
            return cols[normalized_cols.index(cand)]
    return None


SEGMENT_ID_CANDIDATES = [
    "id_tramo", "idtramo", "idtramo", "idlink", "id_link", "segmento", "segment_id", "id", "link_id", "tramo", "tid"
]
SPEED_CANDIDATES = [
    "velocidad", "vel_media", "speed", "velocity", "vel", "kmh", "km_h", "kmph",
    "vel_promedio", "vel_media_ponderada", "vel_ponderada", "vel_media_brt", "vel_media_mixto"
]
DATE_CANDIDATES = ["fecha", "date", "timestamp", "fechahora", "datetime", "fecha_hora", "inicio", "fin"]
TIME_CANDIDATES = ["hora", "time", "cuarto_hora"]
DIST_CANDIDATES = ["distancia", "distance", "dist"]
DIRECTION_CANDIDATES = ["sentido", "direccion", "direction", "dir"]

CAMERA_ID_CANDIDATES = ["id", "camera_id", "id_camara", "codigo", "code"]
CAMERA_DATE_CANDIDATES = ["fecha_instalacion", "install_date", "fecha"]
CAMERA_DIR_CANDIDATES = ["sentido", "direccion", "direction"]

ACC_LAT_CANDIDATES = ["lat", "latitud", "y", "coord_y", "latitude"]
ACC_LON_CANDIDATES = ["lon", "longitud", "x", "coord_x", "longitude", "long"]
ACC_DATE_CANDIDATES = ["fecha", "date", "fechahora", "datetime", "f_evento", "fecha_evento"]
ACC_CITY_CANDIDATES = ["municipio", "ciudad", "municipality", "city"]
ACC_LOCALIDAD_CANDIDATES = ["localidad", "locality"]


def ensure_crs(gdf: gpd.GeoDataFrame, epsg: int) -> gpd.GeoDataFrame:
    if gdf.crs is None:
        gdf = gdf.set_crs(epsg)
    elif gdf.crs.to_epsg() != epsg:
        gdf = gdf.to_crs(epsg)
    return gdf


def to_distance_crs(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    return ensure_crs(gdf, EPSG_LATLON).to_crs(EPSG_DISTANCE)


def try_parse_datetime(df: pd.DataFrame, date_col: Optional[str], time_col: Optional[str]) -> Optional[pd.Series]:
    if date_col is None and time_col is None:
        return None
    if date_col is not None and time_col is not None:
        combo = df[date_col].astype(str).str.strip() + " " + df[time_col].astype(str).str.strip()
        return pd.to_datetime(combo, errors="coerce", infer_datetime_format=True)
    if date_col is not None:
        return pd.to_datetime(df[date_col], errors="coerce", infer_datetime_format=True)
    return pd.to_datetime(df[time_col], errors="coerce", infer_datetime_format=True)


def within_bbox(gdf: gpd.GeoDataFrame, bbox: Tuple[float, float, float, float]) -> gpd.GeoDataFrame:
    xmin, ymin, xmax, ymax = bbox
    rect = box(xmin, ymin, xmax, ymax)
    gdf_ll = ensure_crs(gdf, EPSG_LATLON)
    return gdf_ll[gdf_ll.intersects(rect)]


def expanded_bbox(bbox: Tuple[float, float, float, float], expand_deg: float = 0.05) -> Tuple[float, float, float, float]:
    xmin, ymin, xmax, ymax = bbox
    return (xmin - expand_deg, ymin - expand_deg, xmax + expand_deg, ymax + expand_deg)



In [4]:
# Load cameras
cams = gpd.read_file(CAMERAS_PATH)
if cams.geometry.is_empty.any():
    cams = cams[~cams.geometry.is_empty]

# If not points, take centroids to get a point per camera
if not all(cams.geometry.geom_type.isin(["Point", "MultiPoint"])):
    cams["geometry"] = cams.geometry.centroid

cams = ensure_crs(cams, EPSG_LATLON)

# Select/rename key columns where possible
cols = list(cams.columns)
cam_id_col = pick_first(cols, CAMERA_ID_CANDIDATES) or "camera_id"
if cam_id_col not in cams.columns:
    cams[cam_id_col] = np.arange(1, len(cams) + 1)
cam_date_col = pick_first(cols, CAMERA_DATE_CANDIDATES)
cam_dir_col = pick_first(cols, CAMERA_DIR_CANDIDATES)

cams = cams.rename(columns={
    cam_id_col: "camera_id",
    **({cam_date_col: "install_date"} if cam_date_col else {}),
    **({cam_dir_col: "camera_direction"} if cam_dir_col else {}),
})

if "install_date" in cams:
    cams["install_date"] = pd.to_datetime(cams["install_date"], errors="coerce", infer_datetime_format=True)

print("Cameras loaded:", len(cams))
print(cams.head(3))


Cameras loaded: 92
   FID  OBJECTID_1       camera_id                             NOMBRE_DEL   LONGITUD   LATITUD                    DIRECCIÓN                               NÚMERO_DE VELOCIDADM  CORREDOR_P  \
0    1           1  DEI-001/CAM-01  AV 1 DE MAYO - AV BOYACA (N-S) C. RAP -74.140267  4.615664  AV BOYACÁ - CL 35 SUR (N-S)  MT-20194000619861 Diciembre 13 de 2019         50  Av. Boyacá   
1    2           2  DEI-001/CAM-02  AV 1 DE MAYO - AV BOYACA (S-N) C. RAP -74.140261  4.615398  AV BOYACÁ - CL 35 SUR (S-N)  MT-20194000619861 Diciembre 13 de 2019         50  Av. Boyacá   
2    3           3  DEI-002/CAM-01   AV BOYACA - AV AMERICAS (N-S) C. RAP -74.137980  4.628593      AV BOYACÁ - CL 5A (N-S)  MT-20194000619861 Diciembre 13 de 2019         50  Av. Boyacá   

            VIA_SECUND camera_direction CALZADA CARRILES LOCALIDAD                 INFRACCION                   geometry  
0  Av. Primero de Mayo    (Norte - Sur)  Rápida        2   Kennedy  C.29 - C.14 - C.35 - D.02  PO

In [5]:
# Load Bitcarrier vectors (segments)
vectors = gpd.read_file(VECTORES_PATH)
if vectors.geometry.is_empty.any():
    vectors = vectors[~vectors.geometry.is_empty]

# Keep line-like geometries only
vectors = vectors[vectors.geometry.geom_type.isin(["LineString", "MultiLineString"])]
vectors = ensure_crs(vectors, EPSG_LATLON)

v_cols = list(vectors.columns)
seg_id_col = pick_first(v_cols, SEGMENT_ID_CANDIDATES) or "segment_id"
if seg_id_col not in vectors.columns:
    vectors[seg_id_col] = np.arange(1, len(vectors) + 1)

vectors = vectors.rename(columns={seg_id_col: "segment_id"})

# Optional: keep a small set of useful fields if present
maybe_fields = [c for c in ["name", "road", "via", "length", "speed_limit", "sentido", "direction"] if c in vectors.columns]
vectors = vectors[["segment_id", *maybe_fields, "geometry"]]

print("Segments loaded:", len(vectors))
print(vectors.head(3))


Segments loaded: 868
   segment_id                                           geometry
0           1  MULTILINESTRING ((-74.02762 4.76463, -74.02759...
1           2  MULTILINESTRING ((-74.06853 4.61703, -74.06818...
2           3  MULTILINESTRING ((-74.08252 4.67468, -74.07931...


In [6]:
# Link cameras to nearby segments using a buffer-based spatial join
cams_dist = to_distance_crs(cams)
vectors_dist = to_distance_crs(vectors)

# Build camera buffers
cam_buffers = cams_dist.copy()
cam_buffers["geometry"] = cam_buffers.geometry.buffer(CAMERA_SEGMENT_BUFFER_M)

cand = gpd.sjoin(vectors_dist[["segment_id", "geometry"]], cam_buffers[["camera_id", "geometry"]], how="inner", predicate="intersects")
# cand columns: index_left (vector), index_right (camera), segment_id, camera_id
cam_points_dist = cams_dist.set_index("camera_id")["geometry"]

# Compute distance from camera point to segment
def _row_distance(row):
    return vectors_dist.geometry.iloc[row.name].distance(cam_points_dist.loc[row["camera_id"]])

# Align indices for distance compute
cand = cand.reset_index().rename(columns={"index": "vector_idx"})
# Compute distances row-wise
cand["distance_m"] = cand.apply(lambda r: vectors_dist.geometry.iloc[r["vector_idx"]].distance(cam_points_dist.loc[r["camera_id"]]), axis=1)

# Rank by distance per camera and keep top-N
cand_sorted = cand.sort_values(["camera_id", "distance_m"])
cam_seg_map = cand_sorted.groupby("camera_id").head(MAX_SEGMENTS_PER_CAMERA)[["camera_id", "segment_id", "distance_m"]].reset_index(drop=True)

print("Camera->segment pairs:", len(cam_seg_map))
cam_seg_map.head(10)


Camera->segment pairs: 280


,camera_id,segment_id,distance_m
0,DEI-001/CAM-01,35,17.839839
1,DEI-001/CAM-01,1001030,17.839839
2,DEI-001/CAM-01,36,31.423269
3,DEI-001/CAM-01,1001249,31.423269
4,DEI-001/CAM-02,36,24.244578
5,DEI-001/CAM-02,1001249,24.244578
6,DEI-001/CAM-02,35,25.477569
7,DEI-001/CAM-02,1001030,25.477569
8,DEI-002/CAM-01,35,13.803692
9,DEI-002/CAM-01,1001026,13.803692


In [7]:
# Load and unify Bitcarrier velocity CSVs (2019-2020)

def load_velocity_csv(path: str) -> Optional[pd.DataFrame]:
    try:
        df = pd.read_csv(path, low_memory=False)
    except Exception as e:
        try:
            df = pd.read_csv(path, low_memory=False, encoding="latin-1")
        except Exception:
            print("Failed to read:", path, e)
            return None
    cols = [c for c in df.columns]
    lower_map = {c: c.lower() for c in cols}
    df = df.rename(columns=lower_map)
    cols = list(df.columns)

    seg_col = pick_first(cols, SEGMENT_ID_CANDIDATES)
    spd_col = pick_first(cols, SPEED_CANDIDATES)
    date_col = pick_first(cols, DATE_CANDIDATES)
    time_col = pick_first(cols, TIME_CANDIDATES)
    dist_col = pick_first(cols, DIST_CANDIDATES)
    dir_col = pick_first(cols, DIRECTION_CANDIDATES)

    # If date is an interval start like 'inicio' or a full datetime, ignore time_col
    if date_col and normalize_text(date_col) in {"inicio", "fin", "fechahora", "fecha_hora", "timestamp"}:
        time_col = None

    # Fallback heuristics if not found
    if seg_col is None:
        # try any column that looks like an id (tid, codigo) but prefer numeric id fields
        for cand in ["tid", "codigo", "id", "segment", "tramo"]:
            if cand in cols:
                seg_col = cand
                break

    if spd_col is None:
        # Prefer weighted mean if present, else promedio
        for cand in ["vel_media_ponderada", "vel_ponderada", "vel_promedio", "vel_media", "velocidad"]:
            if cand in cols:
                spd_col = cand
                break

    if seg_col is None or spd_col is None:
        print("Skipping (missing id/speed):", os.path.basename(path))
        print("Columns:", list(df.columns))
        return None

    ts = try_parse_datetime(df, date_col, time_col)
    out = pd.DataFrame({
        "segment_id": df[seg_col].astype(str).str.strip(),
        "speed_kph": pd.to_numeric(df[spd_col], errors="coerce"),
        "timestamp": ts,
    })
    if dist_col:
        out["distance_m"] = pd.to_numeric(df[dist_col], errors="coerce")
    if dir_col:
        out["direction"] = df[dir_col].astype(str)

    out["source_file"] = os.path.basename(path)
    return out

all_vel = []
for folder in [VEL_DIR_2019, VEL_DIR_2020]:
    if not os.path.isdir(folder):
        continue
    for fn in os.listdir(folder):
        if not fn.lower().endswith(".csv"):
            continue
        p = os.path.join(folder, fn)
        vdf = load_velocity_csv(p)
        if vdf is not None:
            all_vel.append(vdf)

if len(all_vel):
    velocities = pd.concat(all_vel, ignore_index=True)
else:
    velocities = pd.DataFrame(columns=["segment_id", "speed_kph", "timestamp", "distance_m", "direction", "source_file"]) 

# Clean
velocities = velocities.dropna(subset=["segment_id"]).copy()
if "timestamp" in velocities:
    velocities["timestamp"] = pd.to_datetime(velocities["timestamp"], errors="coerce")
velocities["year"] = velocities["timestamp"].dt.year
velocities["month"] = velocities["timestamp"].dt.month

print("Velocities rows (all):", len(velocities))
velocities.head(5)


Velocities rows (all): 48864311


,segment_id,speed_kph,timestamp,distance_m,source_file,year,month
0,1000707,46.399040,2019-04-01,688,Velocidades_Bitcarrier_Abril_2019.csv,2019,4
1,1,42.696880,2019-04-01,17482,Velocidades_Bitcarrier_Abril_2019.csv,2019,4
2,1000728,31.328932,2019-04-01,742,Velocidades_Bitcarrier_Abril_2019.csv,2019,4
3,1000675,35.491238,2019-04-01,484,Velocidades_Bitcarrier_Abril_2019.csv,2019,4
4,1000729,33.783825,2019-04-01,1032,Velocidades_Bitcarrier_Abril_2019.csv,2019,4


In [8]:
# Filter velocities to linked segments
linked_segment_ids = set(cam_seg_map["segment_id"].astype(str).unique())
vel_linked = velocities[velocities["segment_id"].astype(str).isin(linked_segment_ids)].copy()
vel_linked = vel_linked.dropna(subset=["speed_kph", "timestamp"]).copy()

vel_linked.to_parquet(os.path.join(OUTPUT_DIR, "velocities_linked.parquet"), index=False)
vel_linked.to_csv(os.path.join(OUTPUT_DIR, "velocities_linked_sample.csv"), index=False)
print("Velocities linked rows:", len(vel_linked))
vel_linked.head(5)


Velocities linked rows: 5404992


,segment_id,speed_kph,timestamp,distance_m,source_file,year,month
1,1,42.696880,2019-04-01,17482,Velocidades_Bitcarrier_Abril_2019.csv,2019,4
2,1000728,31.328932,2019-04-01,742,Velocidades_Bitcarrier_Abril_2019.csv,2019,4
4,1000729,33.783825,2019-04-01,1032,Velocidades_Bitcarrier_Abril_2019.csv,2019,4
11,1000664,31.508530,2019-04-01,853,Velocidades_Bitcarrier_Abril_2019.csv,2019,4
16,1000727,25.786713,2019-04-01,709,Velocidades_Bitcarrier_Abril_2019.csv,2019,4


In [9]:
# DIAGNOSTIC: Compare address formats between 2019 and 2022
import pandas as pd
import os

acc_dir = "ACCIDENTALIDAD"

# Load 2019 SINIESTROS
xls_2019 = pd.ExcelFile(os.path.join(acc_dir, "SIGAT_ANUARIO_2019.xlsx"), engine='openpyxl')
df_2019 = xls_2019.parse("SINIESTROS", nrows=20)

# Load 2022 SINIESTROS
xls_2022 = pd.ExcelFile(os.path.join(acc_dir, "SIGAT_ANUARIO_2022.xlsx"), engine='openpyxl')
for sheet in xls_2022.sheet_names:
    if 'sin' in sheet.lower():
        df_2022 = xls_2022.parse(sheet, nrows=20)
        print(f"2022 columns: {list(df_2022.columns)[:10]}")
        break

print("\nSample 2019 addresses (format: 'AK 10-CL 11 2'):")
print(df_2019[["Direccion"]].head(15).to_string(index=False))

# Find direccion column in 2022 (might be lowercase or different name)
dir_col_2022 = None
for col in df_2022.columns:
    if 'direcc' in col.lower():
        dir_col_2022 = col
        break

if dir_col_2022:
    print("\n" + "="*60)
    print(f"Sample 2022 addresses (column: '{dir_col_2022}'):")
    print(df_2022[[dir_col_2022]].head(15).to_string(index=False))
else:
    print("\n2022 does not have address column - uses lat/lon directly!")



2022 columns: ['CODIGO_ACCIDENTE', 'FORMULARIO', 'GRAVEDAD', 'CLASE', 'CHOQUE', 'OBJETO_FIJO', 'LONGITUD', 'LATITUD', 'DIRECCION', 'CODIGO_LOCALIDAD']

Sample 2019 addresses (format: 'AK 10-CL 11 2'):
                                 Direccion
                             AK 10-CL 11 2
                              KR 4-CL 29 2
                          CL 78-KR 80J S 2
                           CL 100-KR 67A 2
               AV AVENIDA DEL SUR-CL 59 02
               AV AVENIDA BOYACA-CL 72 S 2
                            KR 45-CL 93 02
                           KR 45A-CL 134 2
        AV AVENIDA DE LAS AMERICAS-KR 45 2
                             CL 19-KR 36 2
                             CL 63-TR 93 2
                AV AVENIDA DEL SUR-CL 63 2
                             KR 17-CL 65 2
                              CL 1-KR 27 2
AV AVENIDA CIUDAD DE VILLAVICENCIO-KR 38 2

Sample 2022 addresses (column: 'DIRECCION'):
                             DIRECCION
                     KR 10

In [10]:
# Load accidentalidad from SINIESTROS sheets (2018-2022)
# Strategy: 2022 has coordinates, 2018-2021 have addresses in different format
# Normalize addresses from all years and use 2022 to build geocoding dictionary

def normalize_address(addr: str) -> str:
    """
    Normalize Colombian street addresses to a standard format for matching.
    Examples:
      'AK 10-CL 11 2' -> 'KR10CL11'
      'KR 103 - CL 23 02' -> 'KR103CL23'
      'AV AVENIDA DEL SUR-CL 59 02' -> 'AVSUR CL59'
    """
    if pd.isna(addr) or not addr:
        return ""
    
    s = str(addr).upper().strip()
    
    # Remove common prefixes and normalize abbreviations
    s = s.replace("AVENIDA", "AV")
    s = s.replace("AUTOPISTA", "KR")  # Treat autopista as carrera
    s = s.replace("AK", "KR")  # AK is old notation for KR
    s = s.replace("TRANSVERSAL", "TV")
    s = s.replace("DIAGONAL", "DG")
    
    # Remove specific avenue names but keep the AV prefix
    for term in ["DEL SUR", "BOYACA", "DE LAS AMERICAS", "CIUDAD DE VILLAVICENCIO", 
                 "CARACAS", "NQS", "SUBA", "ESPERANZA"]:
        if term in s:
            # Keep "AV" + simplified name
            s = s.replace(f"AV {term}", "AV")
            s = s.replace(f"AV{term}", "AV")
    
    # Remove all spaces and dashes
    s = s.replace(" ", "").replace("-", "")
    
    # Remove trailing directional indicators (02, 2, S, N, etc.) - these vary between years
    import re
    s = re.sub(r'(0?\d|[NSEO])$', '', s)
    
    # Keep only: street type (KR, CL, AV, etc.) + numbers
    # This creates a canonical form like "KR10CL11"
    return s

def load_siniestros_sheet(path: str, year: int):
    """Load SINIESTROS sheet from accident Excel file"""
    try:
        xls = pd.ExcelFile(path, engine="openpyxl")
        # Find SINIESTROS sheet (case-insensitive)
        siniestros_sheet = None
        for sheet in xls.sheet_names:
            if 'siniestro' in sheet.lower():
                siniestros_sheet = sheet
                break
        
        if siniestros_sheet is None:
            print(f"  No SINIESTROS sheet found in {os.path.basename(path)}")
            return None
        
        df = xls.parse(siniestros_sheet)
        df.columns = [str(c).strip() for c in df.columns]  # Keep original case for now
        df['source_year'] = year
        df['source_file'] = os.path.basename(path)
        return df
    except Exception as e:
        print(f"  Error loading {os.path.basename(path)}: {e}")
        return None

# Load all years
print("Loading accident data from SINIESTROS sheets...")
acc_files = {
    2018: "Base_2018.xlsx",
    2019: "SIGAT_ANUARIO_2019.xlsx",
    2020: "SIGAT_ANUARIO_2020.xlsx",
    2021: "SIGAT_ANUARIO_2021.xlsx",
    2022: "SIGAT_ANUARIO_2022.xlsx",
}

all_siniestros = {}
for year, fn in acc_files.items():
    path = os.path.join(ACC_DIR, fn)
    if not os.path.exists(path):
        print(f"  {year}: file not found")
        continue
    df = load_siniestros_sheet(path, year)
    if df is not None:
        all_siniestros[year] = df
        print(f"  {year}: {len(df)} records from {fn}")

# Step 1: Extract coordinates from 2022 (has lat/lon)
print("\nProcessing 2022 data (has coordinates)...")
acc_2022_frames = []

if 2022 in all_siniestros:
    df_2022 = all_siniestros[2022]
    cols_lower = {c: c.lower() for c in df_2022.columns}
    df_2022_renamed = df_2022.rename(columns=cols_lower)
    
    lat_col = pick_first(list(df_2022_renamed.columns), ACC_LAT_CANDIDATES)
    lon_col = pick_first(list(df_2022_renamed.columns), ACC_LON_CANDIDATES)
    date_col = pick_first(list(df_2022_renamed.columns), ACC_DATE_CANDIDATES)
    
    if lat_col and lon_col:
        tmp_2022 = pd.DataFrame({
            "lat": pd.to_numeric(df_2022_renamed[lat_col], errors="coerce"),
            "lon": pd.to_numeric(df_2022_renamed[lon_col], errors="coerce"),
        })
        if date_col:
            tmp_2022["event_datetime"] = pd.to_datetime(df_2022_renamed[date_col], errors="coerce")
        
        # Keep extra fields
        keep_extra = [c for c in ["direccion", "localidad", "clase", "gravedadnombre"] if c in df_2022_renamed.columns]
        for c in keep_extra:
            tmp_2022[c] = df_2022_renamed[c]
        
        # Filter valid coordinates
        tmp_2022 = tmp_2022.dropna(subset=["lat", "lon"])
        tmp_2022 = tmp_2022[(tmp_2022["lat"] != 0) & (tmp_2022["lon"] != 0)]
        tmp_2022 = tmp_2022[(tmp_2022["lat"] > 3.5) & (tmp_2022["lat"] < 5.5) & (tmp_2022["lon"] > -75) & (tmp_2022["lon"] < -73)]
        
        if len(tmp_2022):
            gtmp_2022 = gpd.GeoDataFrame(tmp_2022, geometry=gpd.points_from_xy(tmp_2022["lon"], tmp_2022["lat"]), crs=f"EPSG:{EPSG_LATLON}")
            acc_2022_frames.append(gtmp_2022)
            print(f"  2022: {len(gtmp_2022)} valid accident points")
        else:
            print(f"  2022: No valid coordinates after filtering")
    else:
        print(f"  2022: No lat/lon columns found")

# Step 2: Build address->coordinates dictionary from 2022 (with normalization)
print("\nBuilding geocoding dictionary from 2022 addresses...")
geocode_dict = {}

if len(acc_2022_frames) and "direccion" in acc_2022_frames[0].columns:
    df_2022_geo = acc_2022_frames[0].copy()
    
    # Normalize addresses to standard format
    df_2022_geo["direccion_normalized"] = df_2022_geo["direccion"].apply(normalize_address)
    
    # Remove empty normalized addresses
    df_2022_geo = df_2022_geo[df_2022_geo["direccion_normalized"] != ""]
    
    # Group by normalized address and take median coordinates (some addresses repeat)
    addr_coords = (df_2022_geo.groupby("direccion_normalized")
                   .agg({"lat": "median", "lon": "median"})
                   .to_dict('index'))
    geocode_dict = {addr: (info["lat"], info["lon"]) for addr, info in addr_coords.items()}
    print(f"  Built dictionary with {len(geocode_dict)} unique normalized addresses")

# Step 3: Process 2018-2021 data using normalized address matching
print("\nProcessing 2018-2021 data (geocoding by normalized address)...")
acc_historical_frames = []

for year in [2018, 2019, 2020, 2021]:
    if year not in all_siniestros:
        continue
    
    df = all_siniestros[year]
    cols_lower = {c: c.lower() for c in df.columns}
    df_renamed = df.rename(columns=cols_lower)
    
    if "direccion" not in df_renamed.columns:
        print(f"  {year}: No 'Direccion' column, skipping")
        continue
    
    date_col = pick_first(list(df_renamed.columns), ["fecha", "date", "timestamp", "fechahora"])
    
    # Prepare data
    tmp = pd.DataFrame({
        "direccion": df_renamed["direccion"].astype(str),
    })
    
    if date_col:
        tmp["event_datetime"] = pd.to_datetime(df_renamed[date_col], errors="coerce")
    
    # Keep extra fields
    keep_extra = [c for c in ["localidad", "clase", "gravedadnombre", "choquenombre"] if c in df_renamed.columns]
    for c in keep_extra:
        tmp[c] = df_renamed[c]
    
    # Normalize addresses and geocode using dictionary
    tmp["direccion_normalized"] = tmp["direccion"].apply(normalize_address)
    tmp["lat"] = tmp["direccion_normalized"].map(lambda addr: geocode_dict.get(addr, (None, None))[0])
    tmp["lon"] = tmp["direccion_normalized"].map(lambda addr: geocode_dict.get(addr, (None, None))[1])
    
    # Filter to valid coordinates
    tmp = tmp.dropna(subset=["lat", "lon"])
    before = len(tmp)
    tmp = tmp[(tmp["lat"] > 3.5) & (tmp["lat"] < 5.5) & (tmp["lon"] > -75) & (tmp["lon"] < -73)]
    
    print(f"  {year}: {len(tmp)} accidents geocoded (from {before} with coords, {len(df)} total)")
    
    if len(tmp):
        gtmp = gpd.GeoDataFrame(tmp, geometry=gpd.points_from_xy(tmp["lon"], tmp["lat"]), crs=f"EPSG:{EPSG_LATLON}")
        acc_historical_frames.append(gtmp)

# Combine all years
print("\nCombining all years...")
all_acc_frames = acc_2022_frames + acc_historical_frames

if len(all_acc_frames):
    accidents = pd.concat(all_acc_frames, ignore_index=True)
else:
    accidents = gpd.GeoDataFrame(columns=["lat", "lon", "event_datetime", "geometry"], geometry="geometry", crs=f"EPSG:{EPSG_LATLON}")

print(f"\nTotal accident points: {len(accidents)}")
if len(accidents) and "event_datetime" in accidents:
    accidents["year"] = accidents["event_datetime"].dt.year
    print("Accidents by year:")
    print(accidents["year"].value_counts().sort_index())

accidents.head(5)


Loading accident data from SINIESTROS sheets...
  No SINIESTROS sheet found in Base_2018.xlsx
  2019: 34990 records from SIGAT_ANUARIO_2019.xlsx
  2020: 22712 records from SIGAT_ANUARIO_2020.xlsx
  2021: 28841 records from SIGAT_ANUARIO_2021.xlsx
  2022: 25447 records from SIGAT_ANUARIO_2022.xlsx

Processing 2022 data (has coordinates)...
  2022: 25447 valid accident points

Building geocoding dictionary from 2022 addresses...
  Built dictionary with 15511 unique normalized addresses

Processing 2018-2021 data (geocoding by normalized address)...
  2019: 15741 accidents geocoded (from 15741 with coords, 34990 total)
  2020: 9676 accidents geocoded (from 9676 with coords, 22712 total)
  2021: 14996 accidents geocoded (from 14996 with coords, 28841 total)

Combining all years...

Total accident points: 65860
Accidents by year:
year
2019    15741
2020     9676
2021    14996
2022    25447
Name: count, dtype: int64


,lat,lon,event_datetime,direccion,localidad,clase,gravedadnombre,geometry,choquenombre,direccion_normalized,year
0,4.682000,-74.13700,2022-07-18 20:44:00,KR 103 - CL 23 02,FONTIBON,1.0,Con Heridos,POINT (-74.137 4.682),NaN,NaN,2022
1,4.612000,-74.08600,2022-04-27 14:30:00,KR 22 - CL 16 02,LOS MARTIRES,1.0,Solo Daños,POINT (-74.086 4.612),NaN,NaN,2022
2,4.694589,-74.10644,2022-04-27 11:30:00,KR 86 - CL 71 A 02,ENGATIVA,1.0,Solo Daños,POINT (-74.10644 4.69459),NaN,NaN,2022
3,4.661000,-74.06200,2022-04-27 19:50:00,CL 73 - AV AVENIDA CARACAS 02,CHAPINERO,1.0,Solo Daños,POINT (-74.062 4.661),NaN,NaN,2022
4,4.627000,-74.08100,2022-04-27 08:00:00,KR 30 - CL 26 02,TEUSAQUILLO,1.0,Solo Daños,POINT (-74.081 4.627),NaN,NaN,2022


In [11]:
# Build coverage areas per camera (buffer around linked segments) and assign accidents

# Prepare distance CRS versions
acc_dist = ensure_crs(accidents, EPSG_LATLON).to_crs(EPSG_DISTANCE)
vec_dist = vectors_dist  # already distance CRS

# Build coverage geometries per camera_id
coverage_rows = []
seg_by_cam = cam_seg_map.groupby("camera_id")["segment_id"].apply(list)
for cam_id, seg_ids in seg_by_cam.items():
    sub = vec_dist[vec_dist["segment_id"].astype(str).isin([str(s) for s in seg_ids])]
    if not len(sub):
        continue
    geom = unary_union(sub.geometry.buffer(COVERAGE_BUFFER_M))
    coverage_rows.append({"camera_id": cam_id, "geometry": geom})

coverage = gpd.GeoDataFrame(coverage_rows, geometry="geometry", crs=f"EPSG:{EPSG_DISTANCE}")
print("Coverage areas:", len(coverage))

# Assign accidents by spatial join
acc_in_cov = gpd.sjoin(acc_dist[["event_datetime", "geometry"]], coverage[["camera_id", "geometry"]], how="inner", predicate="within")
acc_in_cov = acc_in_cov.drop(columns=["index_right"]) if "index_right" in acc_in_cov.columns else acc_in_cov

print("Accidents assigned to cameras:", len(acc_in_cov))
acc_in_cov.head(5)


Coverage areas: 82
Accidents assigned to cameras: 177853


,event_datetime,geometry,camera_id
2,2022-04-27 11:30:00,POINT (996790.012 1010880.081),DEI-011/CAM-01
2,2022-04-27 11:30:00,POINT (996790.012 1010880.081),DEI-011/CAM-02
2,2022-04-27 11:30:00,POINT (996790.012 1010880.081),DEI-020/CAM-01
2,2022-04-27 11:30:00,POINT (996790.012 1010880.081),DEI-020/CAM-02
2,2022-04-27 11:30:00,POINT (996790.012 1010880.081),DEI-031/CAM-01


In [12]:
# Export outputs
cam_seg_out = cam_seg_map.copy()
cam_seg_out.to_csv(os.path.join(OUTPUT_DIR, "camera_segment_mapping.csv"), index=False)

# Save coverage as GeoPackage and GeoJSON
coverage_latlon = coverage.to_crs(EPSG_LATLON)
try:
    coverage_latlon.to_file(os.path.join(OUTPUT_DIR, "camera_coverage.gpkg"), layer="coverage", driver="GPKG")
except Exception as e:
    print("GPKG write failed, writing GeoJSON:", e)
    coverage_latlon.to_file(os.path.join(OUTPUT_DIR, "camera_coverage.geojson"), driver="GeoJSON")

# Accidents linked
acc_linked_latlon = acc_in_cov.to_crs(EPSG_LATLON)
acc_linked_latlon.to_parquet(os.path.join(OUTPUT_DIR, "accidents_linked.parquet"), index=False)
acc_linked_latlon.to_csv(os.path.join(OUTPUT_DIR, "accidents_linked_sample.csv"), index=False)

# Optional: monthly accident counts per camera
if "event_datetime" in acc_linked_latlon:
    acc_linked_latlon["year"] = acc_linked_latlon["event_datetime"].dt.year
    acc_linked_latlon["month"] = acc_linked_latlon["event_datetime"].dt.month
    monthly_acc = (acc_linked_latlon.groupby(["camera_id", "year", "month"]).size().reset_index(name="accidents"))
    monthly_acc.to_csv(os.path.join(OUTPUT_DIR, "accidents_by_camera_month.csv"), index=False)

print("Exported outputs to:", os.path.abspath(OUTPUT_DIR))


Exported outputs to: c:\Users\davib\Downloads\Camaras Salvavidas\outputs


In [13]:
# Quick QA
print({
    "num_cameras": int(len(cams)),
    "num_segments": int(len(vectors)),
    "num_cam_seg_pairs": int(len(cam_seg_map)),
    "vel_rows_linked": int(len(vel_linked)),
    "acc_points": int(len(accidents)),
    "acc_linked": int(len(acc_in_cov)),
})

print("\nSample camera->segments:")
display(cam_seg_map.head(10))

print("\nSample velocities linked:")
display(vel_linked.head(10))

print("\nSample accidents linked:")
display(acc_linked_latlon.head(10))


{'num_cameras': 92, 'num_segments': 868, 'num_cam_seg_pairs': 280, 'vel_rows_linked': 5404992, 'acc_points': 65860, 'acc_linked': 177853}

Sample camera->segments:


,camera_id,segment_id,distance_m
0,DEI-001/CAM-01,35,17.839839
1,DEI-001/CAM-01,1001030,17.839839
2,DEI-001/CAM-01,36,31.423269
3,DEI-001/CAM-01,1001249,31.423269
4,DEI-001/CAM-02,36,24.244578
5,DEI-001/CAM-02,1001249,24.244578
6,DEI-001/CAM-02,35,25.477569
7,DEI-001/CAM-02,1001030,25.477569
8,DEI-002/CAM-01,35,13.803692
9,DEI-002/CAM-01,1001026,13.803692



Sample velocities linked:


,segment_id,speed_kph,timestamp,distance_m,source_file,year,month
1,1,42.696880,2019-04-01,17482,Velocidades_Bitcarrier_Abril_2019.csv,2019,4
2,1000728,31.328932,2019-04-01,742,Velocidades_Bitcarrier_Abril_2019.csv,2019,4
4,1000729,33.783825,2019-04-01,1032,Velocidades_Bitcarrier_Abril_2019.csv,2019,4
11,1000664,31.508530,2019-04-01,853,Velocidades_Bitcarrier_Abril_2019.csv,2019,4
16,1000727,25.786713,2019-04-01,709,Velocidades_Bitcarrier_Abril_2019.csv,2019,4
17,35,43.763176,2019-04-01,20204,Velocidades_Bitcarrier_Abril_2019.csv,2019,4
34,1000730,38.335516,2019-04-01,1029,Velocidades_Bitcarrier_Abril_2019.csv,2019,4
41,40,36.167486,2019-04-01,10453,Velocidades_Bitcarrier_Abril_2019.csv,2019,4
53,75,39.573357,2019-04-01,14695,Velocidades_Bitcarrier_Abril_2019.csv,2019,4
57,77,42.568857,2019-04-01,13450,Velocidades_Bitcarrier_Abril_2019.csv,2019,4



Sample accidents linked:


,event_datetime,geometry,camera_id,year,month
2,2022-04-27 11:30:00,POINT (-74.10644 4.69459),DEI-011/CAM-01,2022,4
2,2022-04-27 11:30:00,POINT (-74.10644 4.69459),DEI-011/CAM-02,2022,4
2,2022-04-27 11:30:00,POINT (-74.10644 4.69459),DEI-020/CAM-01,2022,4
2,2022-04-27 11:30:00,POINT (-74.10644 4.69459),DEI-020/CAM-02,2022,4
2,2022-04-27 11:30:00,POINT (-74.10644 4.69459),DEI-031/CAM-01,2022,4
2,2022-04-27 11:30:00,POINT (-74.10644 4.69459),DEI-031/CAM-02,2022,4
2,2022-04-27 11:30:00,POINT (-74.10644 4.69459),DEI-037/CAM-01,2022,4
2,2022-04-27 11:30:00,POINT (-74.10644 4.69459),DEI-037/CAM-02,2022,4
4,2022-04-27 08:00:00,POINT (-74.081 4.627),DEI-012/CAM-01,2022,4
4,2022-04-27 08:00:00,POINT (-74.081 4.627),DEI-012/CAM-02,2022,4


In [14]:
# Export unified CSV with all accidents and their camera/segment relationships

print("Building unified accidents dataset...")

# Start with accidents linked to cameras (already has camera_id, event_datetime, geometry)
acc_unified = acc_linked_latlon.copy()

# Add year/month if not already present
if "year" not in acc_unified.columns and "event_datetime" in acc_unified.columns:
    acc_unified["year"] = acc_unified["event_datetime"].dt.year
if "month" not in acc_unified.columns and "event_datetime" in acc_unified.columns:
    acc_unified["month"] = acc_unified["event_datetime"].dt.month

# Add camera information (install date, direction, location)
cam_info = cams_dates[["camera_id", "install_date"]].copy()
if "camera_direction" in cams.columns:
    cam_info = cam_info.merge(cams[["camera_id", "camera_direction"]], on="camera_id", how="left")
if "LOCALIDAD" in cams.columns:
    cam_info = cam_info.merge(cams[["camera_id", "LOCALIDAD"]], on="camera_id", how="left")

acc_unified = acc_unified.merge(cam_info, on="camera_id", how="left")

# Add segment information for each camera
# Join with camera-segment mapping to get all related segments
cam_seg_info = cam_seg_map[["camera_id", "segment_id", "distance_m"]].copy()
acc_with_segments = acc_unified.merge(cam_seg_info, on="camera_id", how="left")

# Add pre/post flag
if "install_date" in acc_with_segments.columns and "event_datetime" in acc_with_segments.columns:
    acc_with_segments["is_pre_installation"] = (
        acc_with_segments["install_date"].notna() & 
        (acc_with_segments["event_datetime"] < acc_with_segments["install_date"])
    )
    acc_with_segments["is_post_installation"] = (
        acc_with_segments["install_date"].notna() & 
        (acc_with_segments["event_datetime"] >= acc_with_segments["install_date"])
    )

# Extract lat/lon from geometry for easier use
if "geometry" in acc_with_segments.columns:
    acc_with_segments["lat"] = acc_with_segments.geometry.y
    acc_with_segments["lon"] = acc_with_segments.geometry.x
    # Drop geometry column for CSV export
    acc_with_segments = acc_with_segments.drop(columns=["geometry"])

# Select and order columns for export
export_cols = [
    "camera_id",
    "segment_id", 
    "distance_m",
    "event_datetime",
    "year",
    "month",
    "lat",
    "lon",
    "install_date",
    "is_pre_installation",
    "is_post_installation"
]

# Add optional columns if they exist
optional_cols = ["camera_direction", "LOCALIDAD", "localidad", "clase", "gravedadnombre", "direccion"]
for col in optional_cols:
    if col in acc_with_segments.columns and col not in export_cols:
        export_cols.append(col)

# Keep only columns that exist
export_cols = [c for c in export_cols if c in acc_with_segments.columns]

acc_export = acc_with_segments[export_cols].copy()

# Sort by camera, date
acc_export = acc_export.sort_values(["camera_id", "event_datetime"])

# Export to CSV
unified_csv_path = os.path.join(OUTPUT_DIR, "accidents_unified.csv")
acc_export.to_csv(unified_csv_path, index=False)

print(f"\nUnified accidents CSV exported:")
print(f"  Path: {os.path.abspath(unified_csv_path)}")
print(f"  Rows: {len(acc_export):,}")
print(f"  Columns: {len(acc_export.columns)}")
print(f"\nColumn list: {list(acc_export.columns)}")

# Show sample
print(f"\nSample data (first 5 rows):")
acc_export.head(5)


Building unified accidents dataset...


NameError: name 'cams_dates' is not defined

In [ ]:
# Build per camera-segment metrics and export GeoJSON

# Ensure install_date exists; try to infer from any text field if missing
cams_dates = cams.copy()
if "install_date" not in cams_dates.columns:
    cams_dates["install_date"] = pd.NaT
    import re
    months = {
        "enero": 1, "febrero": 2, "marzo": 3, "abril": 4, "mayo": 5, "junio": 6,
        "julio": 7, "agosto": 8, "septiembre": 9, "setiembre": 9, "octubre": 10, "noviembre": 11, "diciembre": 12
    }
    def parse_spanish_date(text: str):
        s = str(text).strip().lower()
        m = re.search(r"(enero|febrero|marzo|abril|mayo|junio|julio|agosto|septiembre|setiembre|octubre|noviembre|diciembre)\s+(\d{1,2})\s+de\s+(\d{4})", s)
        if m:
            month = months[m.group(1)]
            day = int(m.group(2))
            year = int(m.group(3))
            try:
                return pd.Timestamp(year, month, day)
            except Exception:
                return pd.NaT
        m2 = re.search(r"(\d{1,2})[/-](\d{1,2})[/-](\d{4})", s)
        if m2:
            d, mth, y = int(m2.group(1)), int(m2.group(2)), int(m2.group(3))
            try:
                return pd.Timestamp(y, mth, d)
            except Exception:
                return pd.NaT
        m3 = re.search(r"(\d{4})[/-](\d{1,2})[/-](\d{1,2})", s)
        if m3:
            y, mth, d = int(m3.group(1)), int(m3.group(2)), int(m3.group(3))
            try:
                return pd.Timestamp(y, mth, d)
            except Exception:
                return pd.NaT
        return pd.NaT
    obj_cols = [c for c in cams_dates.columns if cams_dates[c].dtype == object]
    if len(obj_cols):
        inferred = []
        for _, row in cams_dates[obj_cols].iterrows():
            found = pd.NaT
            for c in obj_cols:
                dt = parse_spanish_date(row[c])
                if pd.notna(dt):
                    found = dt
                    break
            inferred.append(found)
        cams_dates["install_date"] = inferred

# Base pairs with geometry and install date
pairs = (cam_seg_map
    .merge(vectors[["segment_id", "geometry"]], on="segment_id", how="left")
    .merge(cams_dates[["camera_id", "install_date"]], on="camera_id", how="left")
)
pairs_gdf = gpd.GeoDataFrame(pairs, geometry="geometry", crs=f"EPSG:{EPSG_LATLON}")
pairs_gdf["segment_id"] = pairs_gdf["segment_id"].astype(str)

# Accident counts pre/post installation per camera
acc_base = acc_in_cov[["camera_id", "event_datetime"]].copy()
acc_base = acc_base.merge(cams_dates[["camera_id", "install_date"]], on="camera_id", how="left")
acc_base = acc_base.dropna(subset=["event_datetime"])  # require event time

acc_base["is_post"] = acc_base["install_date"].notna() & (acc_base["event_datetime"] >= acc_base["install_date"]) 
acc_base["is_pre"]  = acc_base["install_date"].notna() & (acc_base["event_datetime"] <  acc_base["install_date"]) 

acc_agg = (acc_base.groupby("camera_id")
    .agg(accidents_pre=("is_pre", "sum"), accidents_post=("is_post", "sum"))
    .reset_index()
)

# Speed averages pre/post installation per camera-segment
cam_map_for_merge = cam_seg_map[["camera_id", "segment_id"]].copy()
cam_map_for_merge["segment_id"] = cam_map_for_merge["segment_id"].astype(str)

vel_cam = vel_linked.copy()
vel_cam["segment_id"] = vel_cam["segment_id"].astype(str)
vel_cam = vel_cam.merge(cam_map_for_merge, on="segment_id", how="inner")
vel_cam = vel_cam.merge(cams_dates[["camera_id", "install_date"]], on="camera_id", how="left")

vel_cam["is_post"] = vel_cam["install_date"].notna() & (vel_cam["timestamp"] >= vel_cam["install_date"]) 
vel_cam["is_pre"]  = vel_cam["install_date"].notna() & (vel_cam["timestamp"] <  vel_cam["install_date"]) 

vel_cam["speed_pre"]  = np.where(vel_cam["is_pre"], vel_cam["speed_kph"], np.nan)
vel_cam["speed_post"] = np.where(vel_cam["is_post"], vel_cam["speed_kph"], np.nan)

spd_agg = (vel_cam.groupby(["camera_id", "segment_id"])
    .agg(
        speed_pre_kph=("speed_pre", "mean"),
        speed_post_kph=("speed_post", "mean"),
        n_pre_obs=("speed_pre", "count"),
        n_post_obs=("speed_post", "count"),
    )
    .reset_index()
)

# Assemble metrics on pairs (align types)
pairs_gdf["segment_id"] = pairs_gdf["segment_id"].astype(str)
spd_agg["segment_id"] = spd_agg["segment_id"].astype(str)

pairs_metrics = pairs_gdf.merge(acc_agg, on="camera_id", how="left").merge(spd_agg, on=["camera_id", "segment_id"], how="left")

# Optional: round numeric columns for readability
for col in ["distance_m", "speed_pre_kph", "speed_post_kph"]:
    if col in pairs_metrics:
        pairs_metrics[col] = pairs_metrics[col].astype(float).round(3)

# Write GeoJSON
metrics_geojson = os.path.join(OUTPUT_DIR, "camera_segment_metrics.geojson")
pairs_metrics.to_file(metrics_geojson, driver="GeoJSON")
print("Wrote:", os.path.abspath(metrics_geojson))

# Also save as GPKG layer
try:
    pairs_metrics.to_file(os.path.join(OUTPUT_DIR, "camera_segment_metrics.gpkg"), layer="metrics", driver="GPKG")
except Exception as e:
    print("GPKG write failed:", e)
